In [1]:
import os
import matplotlib.pyplot as plt
from src.utils.data import load_json_and_get_model_name
import numpy as np

In [2]:
data_dir = 'outputs/inference/datasets'
output_dir = 'outputs/inference/datasets'

In [3]:
all_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.json')]

In [4]:
metadata = {}

In [5]:
for file_path in all_files:
    data, model_name = load_json_and_get_model_name(file_path)

    if model_name.startswith("ranking_"):
        model_name = model_name.replace("ranking_", "")

    if model_name.startswith("distil_"):
        model_name = model_name.replace("distil_", "")

    metadata[model_name] = data

loaded dataset outputs/inference/datasets/distil-metadata.json
loaded dataset outputs/inference/datasets/ranking_random-metadata.json
loaded dataset outputs/inference/datasets/clap-metadata.json
loaded dataset outputs/inference/datasets/ranking_hard-metadata.json
loaded dataset outputs/inference/datasets/distil_projected-metadata.json


In [6]:
metadata.keys()

dict_keys(['distil', 'random', 'clap', 'hard', 'projected'])

In [7]:
model_colors = {
        'hard': 'tab:blue',
        'random': 'tab:cyan',
        'distil': 'tab:orange',
        'baseline': 'tab:green',
        'clap': 'tab:red',
    }

metrics_to_plot = [
        'time',
        'peak_memory',
        'avg_memory',
        'peak_util',
        'avg_util'
    ]

In [10]:
for metric_name in metrics_to_plot:
    plt.figure(figsize=(8, 5))

    models_for_plot = []
    means_for_plot = []
    stds_for_plot = []
    colors_for_plot = []

    for model, runs in metadata.items():
        values = [run[metric_name] for run in runs if metric_name in run]
        if values:
            mean_val = np.mean(values)
            std_val = np.std(values)
            models_for_plot.append(model)
            means_for_plot.append(mean_val)
            stds_for_plot.append(std_val)
            colors_for_plot.append(model_colors.get(model, 'gray'))

    # sort models alphabetically
    clap_index = models_for_plot.index('clap')
    clap_model = models_for_plot.pop(clap_index)
    clap_mean = means_for_plot.pop(clap_index)
    clap_std = stds_for_plot.pop(clap_index)
    clap_color = colors_for_plot.pop(clap_index)

    sorted_indices = np.argsort(models_for_plot)
    models_for_plot = [models_for_plot[i] for i in sorted_indices]
    means_for_plot = [means_for_plot[i] for i in sorted_indices]
    stds_for_plot = [stds_for_plot[i] for i in sorted_indices]
    colors_for_plot = [colors_for_plot[i] for i in sorted_indices]

    models_for_plot = [clap_model] + models_for_plot
    means_for_plot = [clap_mean] + means_for_plot
    stds_for_plot = [clap_std] + stds_for_plot
    colors_for_plot = [clap_color] + colors_for_plot

    bars = plt.bar(models_for_plot, means_for_plot, yerr=stds_for_plot, capsize=5,
                   color=colors_for_plot, edgecolor='black')

    plt.title(f'{metric_name} across models')
    plt.xlabel("Model", fontsize=14)

    ylabel = 'Vram' if 'memory' in metric_name else 'Time' if 'time' in metric_name else 'Utilization'
    plt.ylabel(ylabel, fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=14)
    plt.grid(True, axis='y', linestyle='--', alpha=0.7)

    # Annotate bars with mean ± std
    for bar, mean, std in zip(bars, means_for_plot, stds_for_plot):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_y() + 0.05 * max(means_for_plot),
                 f'{mean:.1f}±{std:.1f}', ha='center', va='bottom', fontsize=11, color='black')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{metric_name}_barplot.png"))
    plt.close()
